# FORESEE Models: ALP coupling to SU(2)L 

## Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from matplotlib import pyplot as plt

## 1. Specifying the Model

The phenomenology of the ALP coupling to SU(2)$_L$ can be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = - \frac{1}{2} \textcolor{red}{m_{a}}^2 a^2  - \frac{\textcolor{red}{g_{aWW}}}{4}a W^a_{\mu\nu} \tilde W^a_{\mu\nu}
\end{equation}

with the ALP mass $m_a$ and the coupling parameter $g_{aWW}$ as free parameters. For the search for ALPs at forward experiments we need to know i) the *production rate*, and ii) the *interaction rate*. All these properties are specified in the `Model` class. We initialize it with the name of the model as argument. 

In [ ]:
energy = "14"
modelname="ALP-W"
model = Model(modelname, path="./")

# Builder parameters, matching the build.py / load_model() defaults.
nsample_2body = 2000
generators_light = ['EPOSLHC', 'SIBYLL', 'QGSJET'][:1]
generators_heavy = ['NLO-P8', 'NLO-P8-Max', 'NLO-P8-Min'][:1]

**Production** The ALP is mainly produced in FCNC kaon and B-meson decays. The branching fractions are (with $g = g_{aWW}\cdot GeV$)

\begin{equation}
    \text{BR}(K^+ \to \pi^+ a) = 10.5 \times g^2 \times [(1-(m_\pi+m_a)^2/m_K^2)(1-(m_\pi-m_a)^2/m_K^2)]^{1/2}
\end{equation}
\begin{equation}
\text{BR}(K_L \to \pi^0 a) = 4.5 \times g^2 \times [(1-(m_\pi+m_a)^2/m_K^2)(1-(m_\pi-m_a)^2/m_K^2)]^{1/2}
\end{equation}
\begin{equation}
\text{BR}(B \to X_s a)     = 2.3 \cdot 10^4 \times g^2 \times [(1-m_a^2/m_B^2)]^{2}
\end{equation}

In the following, we model light hadron production using `EPOSLHC`, `SIBYLL` and `QGSJET` and heavy hadron production using the `POWHEG+Pythia8` predicions.

In [ ]:
model.add_production_2bodydecay(
    pid0 = "130",
    pid1 = "111",
    br = "4.5 * coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body
)
model.add_production_2bodydecay(
    pid0 = "321",
    pid1 = "211",
    br = "10.5 * coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
)
model.add_production_2bodydecay(
    pid0 = "-321",
    pid1 = "211",
    br = "10.5 * coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body, 
) 

In [ ]:
model.add_production_2bodydecay(
    pid0 = "511",
    pid1 = "130",
    br = "2.3e4 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    pid0 = "-511",
    pid1 = "130",
    br = "2.3e4 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    pid0 = "521",
    pid1 = "321",
    br = "2.3e4 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    pid0 = "-521",
    pid1 = "-321",
    br = "2.3e4 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    pid0 = "531",
    pid1 = "333",
    br = "2.3e4 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    pid0 = "-531",
    pid1 = "333",
    br = "2.3e4 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
)  

**Decay:** The ALP mainly decays to pairs of photons. 

In [ ]:
model.set_ctau_1d(
    filename="model/ctau.txt", 
)

decay_modes = ["gamma_gamma", "e_e_gamma"] 
model.set_br_1d(
    modes = decay_modes,
    finalstates=[[22,22], [11,-11,22]],
    filenames=["model/br/"+mode+".txt" for mode in decay_modes],
)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 2. Event Generation

In the following, we want to study one specific benchmark point with $m_{a}=150$ MeV and $g= 10^{-4}$ and export events as a HEPMC file. 

In [ ]:
mass, coupling, = 0.15, 1e-4

First, we will produce the corresponding flux for this mass and a reference coupling $g_{ref}=1$. 

In [ ]:
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER during 2022/2023. 

In [ ]:
foresee.set_detector(
    distance=474, 
    selection="np.sqrt(x.x**2 + (x.y+0.065)**2)<.1", 
    length=4.0, 
    luminosity=60, 
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames = ['POWHEG-central', 'POWHEG-max', 'POWHEG-min'][:1]

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=None,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses=[round(x,5) for x in np.logspace(-2,np.log10(6.0),50)]
# Extra points around each production channel kinematic endpoint, from
# utility.production_thresholds(model, mass_range).
thresholds = [
    0.34624, 0.35695, 0.36766, 4.21705, 4.34747, 4.47789, 4.6404, 4.78392,
    4.92744,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-8,-3,100) 

# Use cached LLP spectra: get_llp_spectrum recomputes on every call,
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function. Below the production rates, we also show the decay branching fractions of the leading visible final states.

In [ ]:
productions=[
     {"channels": ["130","321","-321"], "color": "blue"   , "label": r"$K \to \pi a$"            , "generators": generators_light},
     {"channels": ["511","-511"]      , "color": "red"    , "label": r"$B^0   \to X_s a + cc.$"  , "generators": generators_heavy},
     {"channels": ["521","-521"]      , "color": "orange" , "label": r"$B^\pm \to X_s a$"        , "generators": generators_heavy},    
     {"channels": ["531","-531"]      , "color": "green"  , "label": r"$B_s   \to X_s a$"        , "generators": generators_heavy},
]

branchings = [
    ["gamma_gamma", "black", "solid", r"$\gamma\gamma$", 0.1, 0.5 ],
    ["e_e_gamma"  , "blue" , "solid", r"$e^+e^-\gamma$"    , 0.1, 0.02],
]

plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",  
    xlims=[0.01,10],ylims=[4e6,4e9],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/g^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(0.97,1),
    fs_label=12,
    ncol=2,
    figsize=(7,6),
    fs_label_br=9,
    branchings=branchings,
)

# one tick per grid mass, pinned to the bottom axis edge
# from matplotlib.transforms import blended_transform_factory
# trans = blended_transform_factory(ax.transData, ax.transAxes)
# ax.plot(masses, [0]*len(masses), marker="|", linestyle="none",
#         color="black", markersize=10, markeredgewidth=0.8,
#         transform=trans, clip_on=False, zorder=5)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")

Let us now scan over various masses and couplings, and record the resulting number of events. Note that here we again consider the FASER configuration, which we set up before.

In [ ]:
setupnames = ['POWHEG-central']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_POWHEG-central.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_POWHEG-central.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_POWHEG-central.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds, separating the bounds obtained by experimental collaboratios and theory recasts. 

In [ ]:
bounds = [   
    ["bounds_BaBar.txt",       "BaBar",    0.25, 7.0e-5, 0       ],
    ["bounds_SN1987.txt",      "SN1987",  0.065, 4e-7, 0       ],
    ["bounds_E137.txt",        "E137",    0.100, 1.1*10**-6, -8  ],
    ["bounds_LEP.txt",         "LEP",     0.650, 6.7*10**-4, 0   ],
    ["bounds_E949_displ.txt",  "E949",    0.065, 9.0*10**-5, -9  ],
    ["bounds_NA62_1.txt",      "NA62",    0.245, 4.5*10**-4, 90  ],
    ["bounds_NA62_2.txt",      "NA62",    0.06, 9.2*10**-6, 2   ],
    ["bounds_KOTO.txt",        "KOTO",    0.070, 3.4*10**-5, 2   ],
    ["bounds_KTEV.txt",        "KTEV",    0.200, 4.5*10**-4, 90  ],
    ["bounds_NA6264.txt",      "+ NA48/2",0.270, 2.5*10**-4, 90  ],
    ["bounds_E949_prompt.txt", "E949",    0.065, 3.0*10**-6, -5  ],
    ["bounds_CDF.txt",         "CDF",     0.065, 6.5*10**-4, -12 ],
]

We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
    # ["limits_Belle2-3gamma.txt",   "royalblue",   r"Belle2 $3\gamma$"+ "\n" +r"50ab$^{-1}$"  , 0.450, 2.1e-4, 0  ],
    # ["limits_KOTO-2gamma.txt",     "cyan",        r"KOTO $2\gamma$"    , 0.090, 2.4e-4, 0  ],
    # ["limits_KOTO-4gamma.txt",     "blue",        r"KOTO $4\gamma$"    , 0.115, 3.2e-4, 0  ],
    # ["limits_NA62-0gamma1.txt",    "dodgerblue",  r"NA62 $0\gamma$"    , 0.220, 2.25e-5, 0  ],
    # ["limits_NA62-0gamma2.txt",    "dodgerblue",  None                 , 0    , 0     , 0  ],
    # ["limits_LHC.txt",             "teal",        r"LHC $Z\to3\gamma$" , 0.650, 5e-6  , 0  ],
]


Finally, we can plot everything using `foresee.plot_reach()`. Here we also add the dark matter relict target line obtained in [2105.07077](https://arxiv.org/abs/2105.07077).

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    projections=projections,
    title="ALPs coupling to SU(2)L", 
    xlims = [0.01,6], 
    ylims=[2e-8,2e-3],    
    xlabel=r"ALP mass $m_{a}$ [GeV]", 
    ylabel=r"ALP coupling $g_{aWW}$ [1/GeV]",
    legendloc=(.82,0.2),
    linewidths=2,
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()